# 13.1 同一問題哪個回答更好？


問「只回數字：2+2=?」，回答A是4，回答B是「4，就像兩雙筷子共有四根」。B更有畫面，但違反只回數字。若問題換成「用生活比喻解釋2+2」，排序又可能改變。本節先把偏好放回同一個任務條件，避免把某種寫法永久貼成好或壞。

前置是[8.5格式與內容分開驗收](https://birdhackor.github.io/tiny-perceptron-vlm/8.5.html)與[8.1風格規則](https://birdhackor.github.io/tiny-perceptron-vlm/8.1.html)。偏好不是說勝者在所有面向都最好，而是按這次明確要求比較。標注者必須知道優先順序：此處先算對，再遵守只回數字；風格不能推翻明確格式。如果不同標注者一個看長短、一個看趣味，訓練就收到混雜訊號。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
pair = {"prompt": "只回數字：2+2=?", "A": "4", "B": "4，就像兩雙筷子共有四根。"}
rule = "先正確，再遵循只回數字"
chosen = "A"
print("條件", pair["prompt"])
print("比較", pair["A"], "／", pair["B"])
print("理由", rule, "選擇", chosen)

輸入同一個提問與兩份人工回答，chosen直接寫A，是人工依據規則做的決定。程式將條件、兩份原文、理由與選擇分別印出，沒有自動讀懂句子來排序。輸出最後「選擇A」應與規則一致：兩份都寫正確數字4，只有A沒有額外文字。

圖中提問只有一份，箭頭分到A與B兩個回答，右側分別列格式結果。兩篇內容都答4，只有A同時符合這次限制。這不是說比喻永遠較差，而是讓偏好理由能對回共同任務。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_preference_pair.svg")))

這樣的資料可讓後來審查者回查選勝者的原因。若只保存一個A勝出標記，就看不見它是因格式而勝、不是因為所有比喻都不好。對安全資料同樣需要[9.1](https://birdhackor.github.io/tiny-perceptron-vlm/9.1.html)的各自標籤：「較好」不能保證本身安全，兩篇都可能還不適合作為理想回答。

偏好也可以有平手或不確定。兩份都正確且滿足格式，又沒有清楚風格差異時，強迫隨便選一個會增加噪聲。先明確比較標準，再決定如何保存平手、是否排除，才能讓訓練目標可解釋。這個小例子故意有明確差異，用來建立最基本的比較單位。

我們真正訓練時先收窄問題，只比較加法真值。例如同一題`2+2=?`，較佳候選是`4`，較差候選是`5`，不是讓模型自訂什麼叫好。兩個加數都從0到7挑選，因此有8×8＝64道題，每題配一組較佳／較差回答。切分時，把交換加數後相同的題目算成一個「家族」：例如`2+3=?`與`3+2=?`必須留在同一組，不能一題拿來教、另一題拿來考，否則測試可能只是在考已見過的數字組合。整個家族一起分配後，本輪共有49對拿來訓練、8對留作驗證，另外7對作最後檢查。訓練組用來更新權重；驗證組用來觀察訓練途中是否學得更好；最後檢查組則留到訓練決定都完成後，檢查未教過的組合。這三組與[8.3的加法起點](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)使用相同問題切分，讓前後比較對得起來。另做一條格式支線，把兩篇都算對的回答拿來比較，見[13.7](https://birdhackor.github.io/tiny-perceptron-vlm/13.7.html)。先分清這兩種理由，後面才知道模型是改了內容排序，還是只偏好沒有附加句的寫法。

練習只移除prompt中的「只回數字」，保留兩篇回答不變。先重新寫下你的偏好理由，再執行打印核對，注意chosen不會自動改變。若你認為B更好，要明確改標記與規則；題目條件改變後重新標注是合理的，但不要在原條件下為了文風偷偷忽略限制。
